# SOC model steps 1-2: minimal bright/dark and spin-Peierls mode

This notebook keeps the model intentionally small. Step 1 uses the same builder as Step 2, but turns off the spin-Peierls mode and keeps only a coherent bright-dark mixing. Step 2 turns on the effective spin-Peierls mode and integrates the response over k.

In [33]:
from pathlib import Path
from datetime import datetime
import json
import sys

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "SolverV8").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate PROJECT_ROOT containing SolverV8.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from SolverV8 import (
    LiouvilleSpectroscopySolver,
    SpectroscopyPlotter,
    standard_nq_protocol,
)


### Parameters

The default numerical values follow the older SOC notebooks: `Delta_dark = 0.90 eV`, `Delta_Bright = 1.50 eV`, `Lambda_SOC = 0.15`, `T = 7 K`, `T_SP_0 = 14 K`, `J = 1`, `delta_0 = 0.01`, and `beta = 0.5`.

For Step 1, the spin-Peierls-specific parameters are set to zero or one trivial k-point. For Step 2, the same model is used with the effective k-dependent spin-Peierls mode turned on.

In [ ]:
base_model_params = {
    # Orbital sector: |g>, |D>, |B>
    "Delta_dark": 0.90,
    "Delta_Bright": 1.0,
    "mu_B": 1.0,
    "mu_D": 0.0,

    # Bright-dark mixing controls
    "V0": 0.0,
    "lambda_delta": 0.0,
    "Lambda_SOC": 0.00,

    # Spin-Peierls effective mode
    "N_k": 100,
    "n_bosons": 2,
    "T": 1.0,
    "T_SP_0": 14.0,
    "B": 0.0,
    "J": 1.0,
    "delta_0": 0.01,
    "beta": 0.5,
    "spin_mode_scale":1.0,

    # k-dependent coupling shape
    "a_dimer": 1.0,
    "alpha_dimerisation": 0.01,
    "parity": "odd",

    # Dissipation, kept explicit so coherent controls are easy
    "gamma_orb": 0.1,
    "gamma_spin": 0.1,
}

step1_params = {
    **base_model_params,
    "label": "step1_minimal_bright_dark",
    "N_k":1,
    "n_bosons": 1,
    "V0": 0.15,
    "lambda_delta": 0.0,
    "Lambda_SOC": 0.0,
    "delta_0": 0.0,
    "spin_mode_scale": 0.0,
    "gamma_orb": 0.0,
    "gamma_spin": 0.0,
}



solver_params = {
    "T": 0.0,
    "Eta": 0.01,
    "backend": "dense",
    "parallel_backend": "threading",
    "n_jobs": -1,
}


### Model Builder

In [35]:
def spin_peierls_delta(T, B, T_SP_0, delta_0, beta):
    alpha_field = 0.004
    T_SP = max(0.0, T_SP_0 * (1.0 - alpha_field * B**2))
    if T_SP > 0.0 and T < T_SP:
        return delta_0 * (1.0 - T / T_SP) ** beta
    return 0.0


def spin_peierls_dispersion(k, T, B, J, T_SP_0, delta_0, beta):
    delta = spin_peierls_delta(T, B, T_SP_0, delta_0, beta)
    gap = 2.0 * J * delta ** (2.0 / 3.0) if delta > 0.0 else 0.0
    velocity = np.pi * J / 2.0
    omega_k = np.sqrt(gap**2 + (velocity * np.sin(k)) ** 2)
    return omega_k, delta, gap


def k_coupling(k, delta, alpha_dimerisation, a_dimer=1.0, parity="odd"):
    phase = k * a_dimer / 2.0
    eta = delta * alpha_dimerisation
    if parity == "odd":
        return np.sin(phase)
    if parity == "even":
        return np.cos(phase)
    return np.sin(phase) + eta * np.cos(phase)


def lowering_operator(n):
    op = np.zeros((n, n), dtype=complex)
    for upper in range(1, n):
        op[upper - 1, upper] = np.sqrt(upper)
    return op


def build_soc_bright_dark_model(params):
    N_k = int(params["N_k"])
    n_bosons = int(params["n_bosons"])
    if N_k == 1:
        k_array = np.array([0.0])
        k_weights = np.ones(1)
    else:
        k_array = np.linspace(-np.pi, np.pi, N_k, endpoint=False)
        k_weights = np.ones(N_k) / N_k

    ket_g, ket_d, ket_b = 0, 1, 2
    H_orb = np.zeros((3, 3), dtype=complex)
    H_orb[ket_d, ket_d] = params["Delta_dark"]
    H_orb[ket_b, ket_b] = params["Delta_Bright"]

    L_bd = np.zeros((3, 3), dtype=complex)
    L_bd[ket_d, ket_b] = 1.0
    L_bd[ket_b, ket_d] = 1.0

    mu_orb = np.zeros((3, 3), dtype=complex)
    mu_orb[ket_g, ket_b] = params["mu_B"]
    mu_orb[ket_b, ket_g] = params["mu_B"]
    mu_orb[ket_g, ket_d] = params["mu_D"]
    mu_orb[ket_d, ket_g] = params["mu_D"]

    a = lowering_operator(n_bosons)
    adag = a.conj().T
    n_op = adag @ a
    x_op = a + adag

    I_orb = np.eye(3, dtype=complex)
    I_spin = np.eye(n_bosons, dtype=complex)
    dim = 3 * n_bosons

    H_stack = np.zeros((N_k, dim, dim), dtype=complex)
    mu_stack = np.zeros_like(H_stack)
    rho0 = np.zeros_like(H_stack)

    for i_k, k in enumerate(k_array):
        omega_k, delta, gap = spin_peierls_dispersion(
            k,
            params["T"],
            params["B"],
            params["J"],
            params["T_SP_0"],
            params["delta_0"],
            params["beta"],
        )
        V_static = params["V0"] + params["lambda_delta"] * delta
        V_k = k_coupling(
            k,
            delta,
            params["alpha_dimerisation"],
            a_dimer=params["a_dimer"],
            parity=params["parity"],
        )

        H_local = np.kron(H_orb, I_spin)
        H_spin = params["spin_mode_scale"] * omega_k * np.kron(I_orb, n_op)
        H_static_mix = V_static * np.kron(L_bd, I_spin)
        H_mode_mix = params["Lambda_SOC"] * V_k * np.kron(L_bd, x_op)

        H_stack[i_k] = H_local + H_spin + H_static_mix + H_mode_mix
        mu_stack[i_k] = np.kron(mu_orb, I_spin)
        rho0[i_k, 0, 0] = 1.0

    c_ops = []
    if params["gamma_spin"]:
        c_ops.append((np.repeat(np.kron(I_orb, a)[None, :, :], N_k, axis=0), params["gamma_spin"]))
    if params["gamma_orb"]:
        C_bg = np.zeros((3, 3), dtype=complex)
        C_bg[ket_g, ket_b] = 1.0
        C_dg = np.zeros((3, 3), dtype=complex)
        C_dg[ket_g, ket_d] = 1.0
        c_ops.append((np.repeat(np.kron(C_bg, I_spin)[None, :, :], N_k, axis=0), params["gamma_orb"]))
        c_ops.append((np.repeat(np.kron(C_dg, I_spin)[None, :, :], N_k, axis=0), params["gamma_orb"]))

    metadata = {
        "k_array": k_array,
        "k_weights": k_weights,
        "delta": spin_peierls_delta(params["T"], params["B"], params["T_SP_0"], params["delta_0"], params["beta"]),
        "dim": dim,
    }
    return H_stack, mu_stack, c_ops, rho0, metadata


### Solver Setup

In [36]:
def make_solver(params):
    H, mu, c_ops, rho0, meta = build_soc_bright_dark_model(params)

    solver = LiouvilleSpectroscopySolver(solver_params)
    solver.feed_model(
        H,
        mu,
        c_ops_raw=c_ops,
        initial_density_matrix=rho0,
        density_matrix_basis="site",
    )
    return solver, meta


solver_params["Eta"] = 0.001
active_params = {
    **step1_params,
    "V0": 0.01,
    "N_k": 1,
}

solver, meta = make_solver(active_params)
print(active_params["label"])
print("Hilbert dimension:", meta["dim"])
print("N_k:", len(meta["k_array"]))
print("delta:", meta["delta"])
print("Hamiltonien dimensions:", solver.H_eigen.shape)


--- Model loading ---
Model transformed to the eigenbasis.
Liouville backend ready: dense.
step1_minimal_bright_dark
Hilbert dimension: 3
N_k: 1
delta: 0.0
Hamiltonien dimensions: (1, 3, 3)


### Third-Order 1Q Pathways With NQ Protocol

In [37]:
arrival_times = [0.0, 100.0, 200.0]
pathways = solver.configure_standard_2d_pathways_with_ufss(arrival_times)

protocol = standard_nq_protocol(
    order=1,
    nq_interval=1,
    detection_interval=3,
    n_interactions=3,
    nq_axis="omega1",
    detection_axis="omega3",
)

rephasing_pathways = solver.get_pathways("rephasing")
unrephasing_pathways = solver.get_pathways("unrephasing")

[(p.name, p.component, p.interactions, p.coherence_orders) for p in pathways]


[('R1', 'rephasing', ('Bu', 'Ku', 'Bd'), (-1, 0, 1)),
 ('R2', 'rephasing', ('Bu', 'Bd', 'Ku'), (-1, 0, 1)),
 ('R3', 'rephasing', ('Bu', 'Ku', 'Ku'), (-1, 0, 1)),
 ('R4', 'unrephasing', ('Ku', 'Bu', 'Ku'), (1, 0, 1)),
 ('R5', 'unrephasing', ('Ku', 'Bu', 'Bd'), (1, 0, 1)),
 ('R6', 'unrephasing', ('Ku', 'Kd', 'Ku'), (1, 0, 1))]

### Spectrum Calculation

The `omega1` axis spans positive and negative 1Q frequencies because the NQ helper separates `+1Q` and `-1Q` contributions from the same protocol.

In [38]:
N_w = 300
omega1_rephasing = np.linspace(-1.1, -0.8, N_w)
omega1_unrephasing = np.linspace(0.8, 1.1, N_w)
omega1 = omega1_rephasing  # Backward-compatible default for rephasing-only checks.
omega3 = np.linspace(0.8, 1.1, N_w)
tau2 = 3.0

# Frequency quadrant convention:
# - rephasing: omega1 < 0, omega3 > 0
# - unrephasing: omega1 > 0, omega3 > 0
# The expensive spectrum calculation is run by loop("scenario", [0]) below.


### Result 2.0 workflow

This section implements the three-step result workflow: code generation, 2D spectrum production, and analysis-ready saved artifacts. `Eta` is fixed in `solver_params`; this test does not scan over `Eta`.


In [39]:
RESULT_ROOT = PROJECT_ROOT / "SOC Model" / "Result_2_0"
DATA_DIR = RESULT_ROOT / "Data"
FIGURES_DIR = RESULT_ROOT / "Figures"
ANALYSIS_DIR = RESULT_ROOT / "Analysis"
for directory in (DATA_DIR, FIGURES_DIR, ANALYSIS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

STEP_NUMBER = 2
SOURCE_NOTEBOOK = RESULT_ROOT / "SOC_steps_1_2_minimal_bright_dark.ipynb"
DETECTION_PHASE = np.pi / 2

SCAN_VALUES = {
    "scenario": [
        {
            "scenario": "minimal_bright_dark",
            "params": {
                "V0": 0.01,
                "N_k": 1,
            },
            "primary_token": ("V0", 0.01, "eV"),
        },
    ],
}


def _safe_token(value):
    text = f"{value:g}" if isinstance(value, (float, np.floating)) else str(value)
    return text.replace("-", "m").replace(".", "p").replace(" ", "")


def make_run_id(step_number, scan_name, index, scenario, primary_token):
    param_name, value, unit = primary_token
    return (
        f"step{int(step_number):02d}__{scan_name}-{int(index):03d}__{scenario}__"
        f"{param_name}-{_safe_token(value)}{unit}"
    )


def build_params_for_run(scan_name, index):
    spec = SCAN_VALUES[scan_name][int(index)]
    params = {**step1_params, **spec["params"]}
    params["label"] = spec["scenario"]
    return params, spec


def configure_2d_protocol(solver):
    arrival_times = [0.0, 100.0, 200.0]
    pathways = solver.configure_standard_2d_pathways_with_ufss(arrival_times)
    protocol = standard_nq_protocol(
        order=1,
        nq_interval=1,
        detection_interval=3,
        n_interactions=3,
        nq_axis="omega1",
        detection_axis="omega3",
    )
    return arrival_times, pathways, protocol


def calculate_component_spectrum(params, component):
    solver, meta = make_solver(params)
    arrival_times, pathways, protocol = configure_2d_protocol(solver)
    component_pathways = [pathway for pathway in pathways if pathway.component == component]
    if component == "rephasing":
        omega1_axis = omega1_rephasing
    elif component == "unrephasing":
        omega1_axis = omega1_unrephasing
    else:
        raise ValueError(f"Unsupported component {component!r}")
    result = solver.generate_NQ_spectrum(
        1,
        protocol,
        axes={"omega1": omega1_axis, "omega3": omega3},
        delays={"t2": tau2},
        pathways=component_pathways,
        k_array=meta["k_array"],
        k_weights=meta["k_weights"],
    )
    return result, solver, meta, component_pathways, arrival_times


def save_spectrum_npz(rephasing_result, unrephasing_result, path):
    spectrum_data = {
        "omega1_rephasing": omega1_rephasing,
        "omega1_unrephasing": omega1_unrephasing,
        "omega3": omega3,
        "S_component_rephasing": rephasing_result.components["rephasing"],
        "S_component_unrephasing": unrephasing_result.components["unrephasing"],
    }
    for name, matrix in rephasing_result.pathways.items():
        spectrum_data[f"S_pathway_{name}"] = matrix
    for name, matrix in unrephasing_result.pathways.items():
        spectrum_data[f"S_pathway_{name}"] = matrix
    np.savez_compressed(path, **spectrum_data)


def _view_matrix(matrix, view):
    phased = np.exp(1j * DETECTION_PHASE) * np.asarray(matrix)
    if view == "real":
        return np.real(phased), "Real", "bwr", None
    if view == "imag":
        return np.imag(phased), "Imaginary", "bwr", None
    if view == "abs":
        return np.abs(phased), "Absolute", "magma", 0.0
    raise ValueError(f"Unknown view {view!r}")


def save_component_figure(result, component, view, path):
    if component not in result.components:
        return None
    x_values = np.asarray(result.axis_values[1], dtype=float)
    y_values = np.asarray(result.axis_values[0], dtype=float)
    values, view_label, cmap, absolute_vmin = _view_matrix(result.components[component], view)
    limit = float(np.max(np.abs(values)))
    if limit == 0.0:
        limit = np.finfo(float).eps
    vmin = absolute_vmin if absolute_vmin == 0.0 else -limit
    levels = np.linspace(vmin, limit, 31)

    fig, ax = plt.subplots(figsize=(6.0, 5.0), constrained_layout=True)
    contour = ax.contourf(x_values, y_values, values, levels=levels, cmap=cmap, vmin=vmin, vmax=limit)
    quadrant = "omega1 < 0" if component == "rephasing" else "omega1 > 0"
    ax.set(
        title=f"{component} {view_label} ({quadrant})",
        xlabel="Detection frequency (eV)",
        ylabel="1Q frequency (eV)",
    )
    fig.colorbar(contour, ax=ax, label=f"{view_label} signal")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path


def write_parameter_file(path, run_id, scan_name, index, spec, params, meta, pathways, generated_files):
    token_name, token_value, token_unit = spec["primary_token"]
    lines = [
        f"run_id: {run_id}",
        f"source_notebook: {SOURCE_NOTEBOOK}",
        f"scan_name: {scan_name}",
        f"index: {int(index)}",
        f"scenario: {spec['scenario']}",
        f"value: {token_value} {token_unit}",
        f"timestamp: {datetime.now().isoformat(timespec='seconds')}",
        "",
        "frequency_quadrants:",
        f"  rephasing: omega1_min={float(omega1_rephasing[0])}, omega1_max={float(omega1_rephasing[-1])}, omega3_min={float(omega3[0])}, omega3_max={float(omega3[-1])}",
        f"  unrephasing: omega1_min={float(omega1_unrephasing[0])}, omega1_max={float(omega1_unrephasing[-1])}, omega3_min={float(omega3[0])}, omega3_max={float(omega3[-1])}",
        "",
        "fixed_physical_parameters:",
    ]
    for key, value in params.items():
        lines.append(f"  {key}: {value}")
    lines.extend(["", "solver_parameters:"])
    for key, value in solver_params.items():
        lines.append(f"  {key}: {value}")
    lines.extend([
        "",
        "spectrum_parameters:",
        f"  N_w: {int(N_w)}",
        f"  tau2: {float(tau2)}",
        "",
        "k_integration:",
        f"  N_k: {int(len(meta['k_array']))}",
        f"  k_min: {float(meta['k_array'][0])}",
        f"  k_max: {float(meta['k_array'][-1])}",
        f"  weight_sum: {float(np.sum(meta['k_weights']))}",
        f"  delta: {float(meta['delta'])}",
        "",
        "pathways:",
    ])
    for pathway in pathways:
        lines.append(
            f"  {pathway.name}: component={pathway.component}, "
            f"interactions={pathway.interactions}, coherence_orders={pathway.coherence_orders}"
        )
    lines.extend(["", "generated_files:"])
    for name, file_path in generated_files.items():
        lines.append(f"  {name}: {Path(file_path).relative_to(RESULT_ROOT)}")
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def run_one(scan_name, index):
    params, spec = build_params_for_run(scan_name, index)
    run_id = make_run_id(STEP_NUMBER, scan_name, index, spec["scenario"], spec["primary_token"])

    rephasing_result, solver, meta, rephasing_pathways, arrival_times = calculate_component_spectrum(params, "rephasing")
    unrephasing_result, _, _, unrephasing_pathways, _ = calculate_component_spectrum(params, "unrephasing")
    pathways = rephasing_pathways + unrephasing_pathways

    spectra_file = DATA_DIR / f"{run_id}_spectra.npz"
    parameter_file = DATA_DIR / f"{run_id}_parameters.txt"
    save_spectrum_npz(rephasing_result, unrephasing_result, spectra_file)

    generated_files = {"spectra": spectra_file}
    for result, component in ((rephasing_result, "rephasing"), (unrephasing_result, "unrephasing")):
        for view in ("real", "imag", "abs"):
            figure_file = FIGURES_DIR / f"{run_id}_{component}_{view}.png"
            saved = save_component_figure(result, component, view, figure_file)
            if saved is not None:
                generated_files[f"{component}_{view}"] = saved

    write_parameter_file(parameter_file, run_id, scan_name, index, spec, params, meta, pathways, generated_files)
    generated_files["parameters"] = parameter_file
    print(f"Saved {run_id}")
    return {
        "run_id": run_id,
        "params": params,
        "rephasing_result": rephasing_result,
        "unrephasing_result": unrephasing_result,
        "meta": meta,
        "files": generated_files,
    }


def loop(scan_name, indices=None):
    if scan_name not in SCAN_VALUES:
        raise KeyError(f"Unknown scan_name {scan_name!r}. Available: {list(SCAN_VALUES)}")
    values = SCAN_VALUES[scan_name]
    if indices is None:
        indices = range(len(values))
    outputs = []
    for index in indices:
        index = int(index)
        if index < 0 or index >= len(values):
            raise IndexError(f"{scan_name}[{index}] is outside 0..{len(values) - 1}")
        outputs.append(run_one(scan_name, index))
    return outputs


### Step 2 launch

Run the next cell to produce the default Result 2.0 spectrum set. This test keeps `Eta` fixed and does not perform an `Eta` scan.


In [40]:
# Produce the default spectrum set for this test.
# Re-run with a different index list if more scenarios are added to SCAN_VALUES.
# outputs = loop("scenario", [0])
